# Solution to p19

The one-down/one-up path preserves an `8x8` skip, builds a `4x4` bottleneck, restores `8x8`, and concatenates decoder then skip. The fuse output is passed directly to the head with no prohibited ReLU. Every loss call validates actual fixture fingerprints before mean pixel cross-entropy is computed.

In [ ]:
import importlib.util
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED=20260901
ATOL=1e-5
RTOL=1e-5
TRAIN_IDS=('segmentation-train-00','segmentation-train-01','segmentation-train-02','segmentation-train-03')
HELDOUT_IDS=('segmentation-heldout-00','segmentation-heldout-01')

def resolve_fixture_path():
    candidates=(
        Path("units/B2-021-cross-modal-transformers-vision/data/vision_fixture.py"),
        Path("../data/vision_fixture.py"),
    )
    existing=tuple(path for path in candidates if path.is_file())
    if len(existing)!=1: raise FileNotFoundError(f"expected exactly one local vision fixture, found: {existing}")
    return existing[0]

fixture_path=resolve_fixture_path()
fixture_spec=importlib.util.spec_from_file_location("p19_vision_fixture",fixture_path)
vision_fixture=importlib.util.module_from_spec(fixture_spec)
sys.modules[fixture_spec.name]=vision_fixture
fixture_spec.loader.exec_module(vision_fixture)

def seed_everything():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1); torch.use_deterministic_algorithms(True)

def _to_torch(fields): return {name:torch.from_numpy(value.copy()) for name,value in fields.items()}
def _to_numpy(fields):
    converted={}
    for name,value in fields.items():
        if not isinstance(value,torch.Tensor): raise TypeError("batch fields must be tensors")
        converted[name]=value.detach().cpu().contiguous().numpy().copy()
    return converted

def build_train_batch(ids):
    if tuple(ids)!=TRAIN_IDS: raise ValueError("train IDs must equal immutable stored order")
    features,targets=vision_fixture.build_task_batch('segmentation', tuple(ids))
    return _to_torch(features),_to_torch(targets)

def build_heldout_batch(ids):
    if tuple(ids)!=HELDOUT_IDS: raise ValueError("held-out IDs must equal immutable stored order")
    features,targets=vision_fixture.build_task_batch('segmentation', tuple(ids))
    return _to_torch(features),_to_torch(targets)

def validate_actual(features,targets):
    role,ids=vision_fixture.validate_actual_batch('segmentation',_to_numpy(features),_to_numpy(targets))
    images,masks=features.get("image"),targets.get("mask")
    if images is None or masks is None or images.ndim!=4 or images.shape[1:]!=(1,8,8): raise ValueError("image shape mismatch")
    if masks.shape!=(images.shape[0],8,8): raise ValueError("mask shape mismatch")
    if images.dtype!=torch.float32 or masks.dtype!=torch.int64: raise ValueError("dtype mismatch")
    return role,ids

class TinyUNetSegmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Conv2d(1,4,3,padding=1)
        self.pool=nn.MaxPool2d(2)
        self.bottleneck=nn.Conv2d(4,8,3,padding=1)
        self.decoder=nn.ConvTranspose2d(8,4,2,stride=2)
        self.fuse=nn.Conv2d(8,4,3,padding=1)
        self.head=nn.Conv2d(4,2,1)
    def forward(self,images,return_aux=False):
        skip=F.relu(self.encoder(images))
        bottleneck=F.relu(self.bottleneck(self.pool(skip)))
        decoder=F.relu(self.decoder(bottleneck))
        if decoder.shape[0]!=skip.shape[0] or decoder.shape[2:]!=skip.shape[2:]: raise ValueError("skip mismatch")
        concatenated=torch.cat((decoder,skip),dim=1)
        fused=self.fuse(concatenated)
        logits=self.head(fused)
        if return_aux: return logits,{"skip":skip,"bottleneck":bottleneck,"decoder":decoder,"concatenated":concatenated,"fused":fused}
        return logits

def compute_loss(model,features,targets):
    validate_actual(features,targets)
    logits=model(features["image"])
    if logits.shape!=(features["image"].shape[0],2,8,8): raise ValueError("logit shape mismatch")
    return F.cross_entropy(logits,targets["mask"],reduction="mean")

def train_unet_segmenter(model,batch,optimizer):
    features,targets=batch
    role,ids=validate_actual(features,targets)
    if role!="train" or ids!=TRAIN_IDS: raise ValueError("optimizer batch must be train only")
    trace=[]; model.train()
    for update in range(1,17):
        optimizer.zero_grad(set_to_none=True)
        loss=compute_loss(model,features,targets)
        loss.backward(); optimizer.step()
        trace.append({"update":update,"mean_pixel_ce":float(loss.detach())})
    return tuple(trace)

def hard_foreground_dice(logits,masks,eps=1e-7):
    prediction=logits.argmax(dim=1)
    predicted_fg=prediction==1; true_fg=masks==1
    intersection=(predicted_fg & true_fg).sum(dtype=torch.float32)
    return (2*intersection+eps)/(predicted_fg.sum(dtype=torch.float32)+true_fg.sum(dtype=torch.float32)+eps)

train_batch=build_train_batch(TRAIN_IDS)
heldout_batch=build_heldout_batch(HELDOUT_IDS)
seed_everything()
model=TinyUNetSegmenter()
optimizer=torch.optim.AdamW(model.parameters(),lr=.03,betas=(.9,.999),eps=1e-8,weight_decay=0,amsgrad=False,foreach=False,fused=False)
model.eval()
with torch.no_grad(): heldout_ce_before=float(compute_loss(model,*heldout_batch))
training_trace=train_unet_segmenter(model,train_batch,optimizer)
model.eval()
with torch.no_grad():
    heldout_ce_after=float(compute_loss(model,*heldout_batch))
    heldout_logits,heldout_aux=model(heldout_batch[0]["image"],return_aux=True)
    heldout_dice=float(hard_foreground_dice(heldout_logits,heldout_batch[1]["mask"],eps=1e-7))
certificate={"updates":len(training_trace),"heldout_improvement":heldout_ce_before-heldout_ce_after,"heldout_dice":heldout_dice,"logits_shape":tuple(heldout_logits.shape),"concat_shape":tuple(heldout_aux["concatenated"].shape)}


### Answer check

In [ ]:
assert len(training_trace)==16
assert tuple(row["update"] for row in training_trace)==tuple(range(1,17))
assert certificate["logits_shape"]==(2,2,8,8)
assert certificate["concat_shape"]==(2,8,8,8)
assert certificate["heldout_improvement"]>=.05
assert certificate["heldout_dice"]>=.65
assert set(TRAIN_IDS).isdisjoint(HELDOUT_IDS)
for bad in (TRAIN_IDS[::-1],TRAIN_IDS[:-1],(TRAIN_IDS[0],TRAIN_IDS[0],*TRAIN_IDS[2:]),(*TRAIN_IDS[:-1],'unknown-id'),HELDOUT_IDS):
    try: build_train_batch(bad)
    except ValueError: pass
    else: raise AssertionError("train wrapper accepted invalid IDs")
for bad in (HELDOUT_IDS[::-1],HELDOUT_IDS[:1],(HELDOUT_IDS[0],HELDOUT_IDS[0]),('unknown-id',HELDOUT_IDS[1]),TRAIN_IDS):
    try: build_heldout_batch(bad)
    except ValueError: pass
    else: raise AssertionError("held-out wrapper accepted invalid IDs")
assert torch.isfinite(compute_loss(model,*train_batch))
